In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect('hubspot.db')

# Let pandas auto-detect encoding
#Change to your personal path to the csv files for this to work and read - you only do this once than you will have a local DB on ur machine and can SQl 
fs = pd.read_csv('/Users/Apple/Desktop/Hubspot/data/processed/frustration_signals_trimmed.csv', encoding_errors='replace') 
csat = pd.read_csv('/Users/Apple/Desktop/Hubspot/data/processed/csat.csv', encoding_errors='replace')
nps = pd.read_csv('/Users/Apple/Desktop/Hubspot/data/processed/nps.csv', encoding_errors='replace')
tickets = pd.read_csv('/Users/Apple/Desktop/Hubspot/data/processed/supportTickets.csv', encoding_errors='replace')

# Load into duckdb
conn.execute("CREATE TABLE IF NOT EXISTS frustration_signals AS SELECT * FROM fs")
conn.execute("CREATE TABLE IF NOT EXISTS csat AS SELECT * FROM csat")
conn.execute("CREATE TABLE IF NOT EXISTS nps AS SELECT * FROM nps")
conn.execute("CREATE TABLE IF NOT EXISTS support_tickets AS SELECT * FROM tickets")

conn.execute("SHOW TABLES").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name
0,csat
1,frustration_signals
2,nps
3,support_tickets


In [3]:
#Column Names for all csvs
for table in ['csat', 'frustration_signals', 'nps', 'support_tickets']:
    print(f"\n--- {table} ---")
    print(conn.execute(f"DESCRIBE {table}").df()[['column_name', 'column_type']].to_string(index=False))


--- csat ---
     column_name column_type
            Date     VARCHAR
       Portal ID      DOUBLE
           Score      DOUBLE
     End User ID      DOUBLE
   Response Text     VARCHAR
Survey Name Sort      BIGINT
     Survey Name     VARCHAR
   Taxonomy Type     VARCHAR

--- frustration_signals ---
    column_name column_type
        COUNTRY     VARCHAR
     EVENT_TIME     VARCHAR
    SIGNAL_NAME     VARCHAR
DEPLOYABLE_NAME     VARCHAR
  PAGE_CATEGORY     VARCHAR

--- nps ---
    column_name column_type
    Response ID      BIGINT
           Date     VARCHAR
          Score      BIGINT
  Taxonomy Type     VARCHAR
  Response Text     VARCHAR
Translated Text     VARCHAR
       Page URL     VARCHAR

--- support_tickets ---
       column_name column_type
       History Key     VARCHAR
       Ticket Text     VARCHAR
          Language     VARCHAR
         Owner IDs     VARCHAR
       Ticket Name     VARCHAR
      Product Area     VARCHAR
         Roadblock     VARCHAR
        Resolution

In [ ]:

# CSAT - avg score by survey name
conn.execute("""
    SELECT "Survey Name", AVG("Score") as avg_score, COUNT(*) as responses
    FROM csat
    GROUP BY "Survey Name"
    ORDER BY avg_score DESC
""").df()

,Survey Name,avg_score,responses
0,Academy CSAT,4.472632,16351
1,Payment Links Index,4.380503,318
2,Import Export,4.348695,10958
3,Admin Setup,4.346796,9879
4,Create Custom Objects,4.307692,26
...,...,...,...
66,UI Extension Cards,3.354756,389
67,IAH Experience,3.192582,701
68,CES:Purchase_exp,NaN,1
69,CES:Academy,NaN,1


In [5]:
# FRUSTRATION SIGNALS - top signals by country
conn.execute("""
    SELECT COUNTRY, SIGNAL_NAME, COUNT(*) as count
    FROM frustration_signals
    GROUP BY COUNTRY, SIGNAL_NAME
    ORDER BY count DESC
    LIMIT 20
""").df()

,COUNTRY,SIGNAL_NAME,count
0,United States,"""Rage Click""",1294978
1,United States,"""Repeated Navigation""",1203970
2,United States,"""Text Highlighting""",1063871
3,None,"""Rage Click""",836963
4,United States,"""Doom Scroll""",796611
5,None,"""Repeated Navigation""",768644
6,Philippines,"""Repeated Navigation""",534747
7,None,"""Text Highlighting""",462505
8,United Kingdom,"""Rage Click""",432079
9,None,"""Doom Scroll""",370617


In [6]:
# NPS - score distribution
conn.execute("""
    SELECT Score, COUNT(*) as count
    FROM nps
    GROUP BY Score
    ORDER BY Score
""").df()

,Score,count
0,0,1406
1,1,369
2,2,444
3,3,464
4,4,411
5,5,797
6,6,644
7,7,779
8,8,1244
9,9,1145


In [7]:
# SUPPORT TICKETS - tickets by product area and resolution
conn.execute("""
    SELECT "Product Area", "Resolution", COUNT(*) as count
    FROM support_tickets
    GROUP BY "Product Area", "Resolution"
    ORDER BY count DESC
    LIMIT 20
""").df()

,Product Area,Resolution,count
0,None,None,237465
1,Workflows,None,51251
2,Marketing Email,None,49571
3,Billing,None,42530
4,Account Settings,None,38527
5,Forms,None,31282
6,Reports,None,29824
7,Object,None,27699
8,Content,None,22747
9,Integrations,None,22571
